# Week 13: wavelet compression

**What we do.** We compress a photograph as in the lecture notes: compute its 2D wavelet transform, set the smallest detail coefficients to zero (thresholding) and reconstruct. We compare hard and soft thresholding, compression ratios of 90%, 95% and 98%, and the filters db1, db3 and db5, and finish with a comparison against Fourier band coding.

**What to expect.** Even with 95% of the coefficients set to zero the photograph is easily recognizable; artefacts appear first near edges. Hard thresholding usually gives a higher PSNR than soft thresholding at the same ratio, unless the soft-thresholded details are rescaled. At the same ratio, wavelets beat Fourier band coding because they adapt to edges.

**How to read the code.** As in the notes, the **compression ratio** is the fraction of zero coefficients among all coefficients. Functions come from `wavelets_lab.compression`, `wavelets_lab.filters` (PSNR), `wavelets_lab.images` and `wavelets_lab.fourier` (for the comparison).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from wavelets_lab import compression, filters, fourier, images

## Thresholding rules

**What we do.** We plot the hard and soft thresholding functions for threshold 1.

**What to expect.** Both are 0 on `[-1, 1]`. Hard thresholding keeps larger values unchanged (a jump at ±1, "keep or kill"); soft thresholding shifts them towards 0 by 1 (continuous, "shrink or kill").

**How to read the code.** `compression.hard_threshold(x, 1.0)` and `compression.soft_threshold(x, 1.0)` act elementwise on the array `x`.

In [ ]:
x = np.linspace(-3.0, 3.0, 601)
plt.figure(figsize=(6, 5))
plt.plot(x, x, "k:", label="identity")
plt.plot(x, compression.hard_threshold(x, 1.0), label="hard")
plt.plot(x, compression.soft_threshold(x, 1.0), label="soft")
plt.xlabel("coefficient before thresholding")
plt.ylabel("coefficient after thresholding")
plt.legend()
plt.show()

## Compression ratios

**What we do.** We compress `camera` with db3 (4 levels, hard thresholding) at ratios 90%, 95% and 98%.

**What to expect.** 90%: hardly any difference from the original; 95%: slight blurring of fine texture (grass); 98%: clear blocky and ringing artefacts, yet the scene is still recognizable. The titles give the achieved fraction of zeros and the PSNR.

**How to read the code.** `compression.compress_image(camera, ratio, "db3", 4)` returns the reconstruction and the achieved ratio; `filters.psnr` measures the quality.

In [ ]:
camera = images.load_test_image("camera")
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
for ax, ratio in zip(axes, (0.90, 0.95, 0.98), strict=True):
    result, achieved = compression.compress_image(camera, ratio, "db3", 4)
    images.show_image(
        result,
        f"{achieved:.1%} zeros, PSNR {filters.psnr(camera, result):.1f} dB",
        ax=ax,
    )
plt.tight_layout()
plt.show()

## A closer look at the artefacts

**What we do.** We zoom into a 128×128 detail (the photographer's head and camera) for the original and the 98% compression with db1 and db3.

**What to expect.** db1 (Haar) produces blocky, piecewise constant artefacts; db3 produces smoother artefacts with some ringing near edges, which usually looks more natural.

**How to read the code.** `image[rows, cols]` with slices cuts out the detail; `show_image` labels the axes in pixels of the detail.

In [ ]:
detail = (slice(60, 188), slice(200, 328))
versions = {"original": camera}
for name in ("db1", "db3"):
    versions[f"98% zeros, {name}"] = compression.compress_image(camera, 0.98, name, 4)[
        0
    ]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (title, image) in zip(axes, versions.items(), strict=True):
    images.show_image(image[detail], title, ax=ax)
plt.tight_layout()
plt.show()

## Quality as a function of the compression ratio

**What we do.** We plot the PSNR against the compression ratio for db1, db3 and db5 with hard thresholding.

**What to expect.** The PSNR decreases as more coefficients are zeroed: from about 55 dB at 50% zeros to about 26 dB at 99%. For this photograph the three filters differ by only a few tenths of a dB, db3 being slightly the best: photographs contain both smooth regions (favouring smooth wavelets) and sharp edges and texture (favouring short ones).

**How to read the code.** Two nested loops: over wavelets and over ratios; each point is one call of `compression.compress_image` followed by `filters.psnr`.

In [ ]:
ratios = [0.5, 0.7, 0.8, 0.9, 0.95, 0.98, 0.99]
plt.figure(figsize=(8, 4.5))
for name in ("db1", "db3", "db5"):
    quality = [
        filters.psnr(camera, compression.compress_image(camera, r, name, 4)[0])
        for r in ratios
    ]
    plt.plot(np.array(ratios) * 100, quality, "o-", label=name)
plt.xlabel("compression ratio (zero coefficients) [%]")
plt.ylabel("PSNR [dB]")
plt.legend()
plt.show()

## Filters and thresholding rules

**What we do.** At a fixed ratio of 95% we compare the three filters with hard thresholding, soft thresholding, and soft thresholding with rescaled details (the correction `M / (M - ε)` from the lecture notes).

**What to expect.** Plain soft thresholding has the lowest PSNR: shrinking all kept coefficients reduces contrast. Rescaling repairs part of that loss.

**How to read the code.** `compression.compress_image(image, ratio, wavelet, level, mode, rescale)`; the printed table has one line per combination (`!s:5s` prints the Boolean as text padded to width 5).

In [ ]:
for name in ("db1", "db3", "db5"):
    for mode, rescale in (("hard", False), ("soft", False), ("soft", True)):
        result, _ = compression.compress_image(camera, 0.95, name, 4, mode, rescale)
        psnr = filters.psnr(camera, result)
        print(f"{name} {mode:4s} rescale={rescale!s:5s} PSNR {psnr:.2f} dB")

## Wavelets versus Fourier band coding

**What we do.** We keep about 4.8% of the coefficients (ratio about 95.2%) with Fourier band coding (week 6) and with wavelet compression, and compare the images.

**What to expect.** The Fourier version is blurred everywhere and rings around edges; the wavelet version keeps edges sharp and has a clearly higher PSNR. Wavelet thresholding keeps the large coefficients wherever they are (adaptively), while band coding always keeps the same low frequencies.

**How to read the code.** `fourier.band_code(camera, 56)` keeps a 112×112 square of frequencies; we then ask `compression.compress_image` for the same fraction of zeros, `1 - kept`.

In [ ]:
fourier_result, kept = fourier.band_code(camera, 56)
wavelet_result, achieved = compression.compress_image(camera, 1 - kept, "db3", 4)
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
psnr_fourier = filters.psnr(camera, fourier_result)
psnr_wavelet = filters.psnr(camera, wavelet_result)
title_fourier = f"Fourier: kept {kept:.1%}, PSNR {psnr_fourier:.1f} dB"
title_wavelet = f"wavelets: kept {1 - achieved:.1%}, PSNR {psnr_wavelet:.1f} dB"
images.show_image(fourier_result, title_fourier, ax=axes[0])
images.show_image(wavelet_result, title_wavelet, ax=axes[1])
plt.tight_layout()
plt.show()

## Exercise: computer graphics

**What we do.** We repeat the comparison of filters for the synthetic `checkerboard` image (a computer graphic made of flat squares).

**What to expect.** At 90% zeros db1 (Haar) wins by a huge margin (the reconstruction is practically exact): piecewise constant images are represented very sparsely by piecewise constant wavelets, while longer, smoother wavelets spread every sharp edge over many coefficients and create ringing. Explain the difference to the photograph, and check what happens at 95% and 98%.

**How to read the code.** The same loop as above with another image (200×200, so 3 levels are possible) and a fixed ratio of 90%.

In [ ]:
checker = images.load_test_image("checkerboard")
for name in ("db1", "db3", "db5"):
    result, _ = compression.compress_image(checker, 0.90, name, 3)
    print(f"{name}: PSNR {filters.psnr(checker, result):.2f} dB at 90% zeros")

## Reference: scripts and functions used in this notebook

**What we do.** We list every function of the course package used above with its purpose, input, output and a recipe for writing it yourself.

**What to expect.** Five modules are involved: `wavelets_lab.images` (loading, display), `wavelets_lab.compression` (thresholding and compression), `wavelets_lab.dwt` (the 2D DWT used inside `compress_image`), `wavelets_lab.filters` (PSNR) and `wavelets_lab.fourier` (band coding for the comparison).

**How to read the code.** Functions marked *used internally* are called by `compress_image`.

**Library functions called directly in this notebook.**

| Function | What it does | Input → output |
|---|---|---|
| `slice(a, b)` | the index range `a, …, b-1`; `image[slice_rows, slice_cols]` cuts a rectangle | numbers → slice |
| `np.linspace`, `np.array` | grids and arrays | – |
| `plt.plot(x, y, "o-")` | line with markers | arrays → line |

### Module `wavelets_lab.images` (file `src/wavelets_lab/images.py`)

Images as NumPy arrays and simple pixel operations. **Convention used everywhere in the course:** a greyscale image is a 2D array of type `float64` with values on the scale `[0, 255]` (0 = black, 255 = white); a colour image is a 3D array of shape `(rows, cols, 3)` on the same scale. Working in `float64` (instead of the `uint8` stored in image files) avoids overflow and rounding during computations.

The module starts with the imports
```python
from pathlib import Path
import imageio.v3 as iio
import matplotlib.pyplot as plt
import numpy as np
from skimage import data
```
and the constant `MAX_INTENSITY = 255.0`.

#### `images.load_test_image(name="camera")`

**What it is.** Loads one of the licence-free test images shipped with scikit-image (`skimage.data`), so that no image file is needed.

**Input.**
- `name` (`str`, default `"camera"`): name of a function in `skimage.data`, e.g. `"camera"` (512×512 greyscale), `"astronaut"` (512×512 RGB), `"coins"`, `"checkerboard"`.

**Output.** `numpy.ndarray` of type `float64` on the scale `[0, 255]` (2D for greyscale, `(rows, cols, 3)` for colour). Raises `ValueError` if the name does not exist.

**How to write it yourself.**
1. Import the module of test images: `from skimage import data`.
2. Look the loader up by name without failing: `loader = getattr(data, name, None)`.
3. If `loader` is `None` or not callable (`callable(loader)` is `False`), raise `ValueError(f"skimage.data has no test image called {name!r}")`.
4. Call it: `raw = loader()` returns a `uint8` array.
5. Return `to_float(raw)` (see `images.to_float`), which converts to `float64` on `[0, 255]`.

#### `images.to_float(image)` (used internally)

**What it is.** Converts any image array to the course convention: `float64` on `[0, 255]`.

**Input.**
- `image` (array-like): `uint8`, `bool`, integer or floating-point image.

**Output.** `numpy.ndarray` of type `float64`.

**How to write it yourself.**
1. `array = np.asarray(image)`.
2. If `array.dtype == np.bool_`: return `array.astype(np.float64) * 255.0` (True becomes white).
3. If the dtype is floating (`np.issubdtype(array.dtype, np.floating)`), the array is not empty and all values lie in `[0, 1]` (`array.min() >= 0` and `array.max() <= 1`): the image uses the unit scale, so return `array.astype(np.float64) * 255.0`.
4. Otherwise (integers, or floats already on a large scale) return `array.astype(np.float64)`.

#### `images.show_image(image, title=None, ax=None, scaled=False, xlabel="column [pixel]", ylabel="row [pixel]")`

**What it is.** Displays a greyscale or colour image with Matplotlib, using a 256-level grey colour map and **labelled axes** (by default pixel coordinates: column on the horizontal axis, row on the vertical axis, with row 0 at the top).

**Input.**
- `image` (array-like): 2D greyscale or `(rows, cols, 3)` RGB image on `[0, 255]`.
- `title` (`str` or `None`): title of the plot.
- `ax` (`matplotlib.axes.Axes` or `None`): where to draw; when `None` a new figure is created. Passing `ax` lets you place several images in one figure (`fig, axes = plt.subplots(1, 3)`).
- `scaled` (`bool`, default `False`): `False` maps 0 to black and 255 to white (true intensities); `True` stretches the smallest value of the data to black and the largest to white, like Matlab's `imagesc`. Use `True` for arrays that are not intensities (spectra, gradients, phases).
- `xlabel`, `ylabel` (`str`): axis labels; change them when the axes mean something else (e.g. angle and offset for a sinogram).

**Output.** The `Axes` object that contains the image (useful for further changes such as `ax.set_xlim`).

**How to write it yourself.**
1. `array = np.asarray(image, dtype=np.float64)`.
2. If `ax is None`: `_, ax = plt.subplots()`.
3. If `array.ndim == 3` (colour): `ax.imshow(to_uint8(array))`; Matplotlib then uses the RGB values directly.
4. Otherwise (greyscale): choose limits `vmin, vmax = (None, None)` if `scaled` else `(0.0, 255.0)` and call `ax.imshow(array, cmap=plt.get_cmap("gray", 256), vmin=vmin, vmax=vmax)`. With `vmin=vmax=None` Matplotlib uses the data minimum and maximum.
5. If a title is given: `ax.set_title(title)`.
6. Label both axes: `ax.set_xlabel(xlabel)` and `ax.set_ylabel(ylabel)`.
7. Return `ax`.

#### `images.to_uint8(image)` (used internally)

**What it is.** The inverse direction: prepares an image for saving or for `imshow` of colour images, which expects bytes.

**Input.**
- `image` (array-like): image on the scale `[0, 255]`, possibly with values outside it.

**Output.** `numpy.ndarray` of type `uint8`.

**How to write it yourself.**
1. Convert to floats: `np.asarray(image, dtype=np.float64)`.
2. Round to the nearest integer with `np.rint`.
3. Clip to the valid range with `np.clip(..., 0, 255)`; without this, e.g. 260 would wrap around to 4.
4. Cast with `.astype(np.uint8)` and return.

### Module `wavelets_lab.compression` (file `src/wavelets_lab/compression.py`)

Thresholding of wavelet coefficients and wavelet image compression. As in the lecture notes, the **compression ratio** is the fraction of wavelet coefficients set to zero among all coefficients (for an orthogonal transform there are as many coefficients as pixels). Imports:
```python
import numpy as np
from wavelets_lab.dwt import decompose_image, reconstruct_image
```
Internal helper `_details(coefficients)`: all detail coefficients of a 2D DWT concatenated into one 1D array (`np.concatenate([np.ravel(band) for bands in coefficients[1:] for band in bands])`).

#### `compression.hard_threshold(x, threshold)`

**What it is.** Hard thresholding: values with `|x| < threshold` become 0, the others are kept unchanged ("keep or kill"). Same as `pywt.threshold(x, t, "hard")`.

**Input.** `x` (array-like): coefficients; `threshold` (`float`, non-negative).

**Output.** `float64` array.

**How to write it yourself.** Return `np.where(np.abs(x) >= threshold, x, 0.0)`.

#### `compression.soft_threshold(x, threshold)`

**What it is.** Soft thresholding: small values become 0 and all others are shrunk towards zero by `threshold`, `sign(x) * max(|x| - threshold, 0)` ("shrink or kill"). The result is continuous in `x` (no jump at the threshold), but large coefficients lose `threshold` of their size. Same as `pywt.threshold(x, t, "soft")`.

**Input.** `x` (array-like), `threshold` (`float`, non-negative).

**Output.** `float64` array.

**How to write it yourself.** Return `np.sign(x) * np.maximum(np.abs(x) - threshold, 0.0)`.

#### `compression.compress_image(image, ratio, wavelet="db2", level=4, mode="hard", rescale=False)`

**What it is.** Wavelet compression of an image as in the lecture notes: 2D DWT, zero the smallest detail coefficients so that the requested fraction of zeros is reached, reconstruct. Only the remaining non-zero coefficients would have to be stored.

**Input.**
- `image` (array-like): greyscale image with sides divisible by `2**level`.
- `ratio` (`float`): target fraction of zeros, e.g. `0.9`, `0.95`, `0.98`.
- `wavelet` (`str`), `level` (`int`): transform parameters.
- `mode` (`"hard"` or `"soft"`): thresholding rule.
- `rescale` (`bool`): after soft thresholding, multiply the details by `M / (M - ε)` (`M` = largest absolute detail, `ε` = threshold) to restore their range; the notes recommend this for large thresholds because soft thresholding lowers the contrast.

**Output.** Tuple `(reconstruction, achieved_ratio)`: the decompressed `float64` image and the fraction of zeros actually obtained.

**How to write it yourself.**
1. `coefficients = decompose_image(image, wavelet, level)`.
2. `threshold = threshold_for_ratio(coefficients, ratio)`.
3. `compressed = threshold_details(coefficients, threshold, mode)`.
4. If `rescale` and `mode == "soft"`: `M = np.max(np.abs(_details(coefficients)))`; if `M > threshold`, multiply every detail band of `compressed` by `M / (M - threshold)`.
5. `achieved = zero_fraction(compressed)`.
6. Return `reconstruct_image(compressed, wavelet), achieved`.

#### `compression.threshold_for_ratio(coefficients, ratio)` (used internally)

**What it is.** Finds the threshold that zeroes a requested fraction (e.g. 90%) of all coefficients. Since only details are thresholded, the threshold is an order statistic of the absolute detail values.

**Input.** `coefficients` (list from `decompose_image`), `ratio` (`float` in `[0, 1)`).

**Output.** `float`: the threshold. Raises `ValueError` outside `[0, 1)`.

**How to write it yourself.**
1. Check `0 <= ratio < 1`.
2. Sort the absolute details: `details = np.sort(np.abs(_details(coefficients)))`.
3. Total number of coefficients: `total = details.size + coefficients[0].size`.
4. Number of coefficients to zero: `count = min(ceil(ratio * total), details.size)`.
5. If `count == 0` return `0.0`; otherwise return `details[count - 1]` (the `count`-th smallest absolute value).

#### `compression.threshold_details(coefficients, threshold, mode="hard")` (used internally)

**What it is.** Applies hard or soft thresholding to every detail subband of a 2D DWT, leaving the approximation `cA_J` untouched (it holds the coarse picture and is never thrown away).

**Input.** `coefficients` (list from `decompose_image`), `threshold` (`float`), `mode` (`"hard"` or `"soft"`).

**Output.** List in the same format. Raises `ValueError` for another mode.

**How to write it yourself.**
1. `rules = {"hard": hard_threshold, "soft": soft_threshold}`; raise `ValueError` if `mode` is not a key.
2. Start the result with a copy of `coefficients[0]`.
3. For each tuple `(cH, cV, cD)` in `coefficients[1:]`, append `(rule(cH, t), rule(cV, t), rule(cD, t))`.

#### `compression.zero_fraction(coefficients)` (used internally)

**What it is.** The fraction of all coefficients (approximation and details) that are exactly zero, i.e. the achieved compression ratio.

**Input.** `coefficients` (list from `decompose_image`).

**Output.** `float` in `[0, 1]`.

**How to write it yourself.**
1. `approximation = np.ravel(coefficients[0])`, `details = _details(coefficients)`.
2. Count zeros with `np.count_nonzero(array == 0)` in both.
3. Return the total count divided by `approximation.size + details.size`.

### Module `wavelets_lab.dwt` (file `src/wavelets_lab/dwt.py`)

The Mallat algorithm (discrete wavelet transform, DWT) in 1D, written from scratch, and the 2D DWT of images via PyWavelets. Signals are treated as **periodic**, which makes the transform orthogonal (it preserves energy) and matches `mode="periodization"` of PyWavelets; then a signal of length `N` gives `N/2` approximation and `N/2` detail coefficients. Imports:
```python
import numpy as np
import pywt
```
Two internal helpers: `_decomposition_filters(wavelet)` returns `pywt.Wavelet(wavelet).dec_lo` and `.dec_hi` as arrays (`dec_lo` is `rec_lo` reversed); `_analysis_indices(n, length)` returns the integer matrix `idx[k, m] = (2k + length // 2 - m) mod n` for `k = 0, …, n/2 - 1` and `m = 0, …, length - 1`, i.e. which sample is multiplied by filter tap `m` in output `k`.

#### `dwt.decompose_image(image, wavelet="db2", level=3)`

**What it is.** The 2D DWT of an image with `pywt.wavedec2` (periodization). One level filters the rows and then the columns, giving four subbands: the approximation `cA` (low-pass in both directions) and the details `cH` (horizontal edges), `cV` (vertical edges) and `cD` (diagonal); the next level decomposes `cA` again.

**Input.** `image` (array-like): greyscale image with sides divisible by `2**level`; `wavelet` (`str`), e.g. `"db1"`, `"db3"`, `"db5"`; `level` (`int`).

**Output.** List `[cA_J, (cH_J, cV_J, cD_J), …, (cH_1, cV_1, cD_1)]`.

**How to write it yourself.**
1. Call `pywt.wavedec2(np.asarray(image, dtype=np.float64), wavelet, mode="periodization", level=level)`.
2. Return the result as a list.

#### `dwt.reconstruct_image(coefficients, wavelet="db2")`

**What it is.** The inverse 2D DWT (`pywt.waverec2` with periodization).

**Input.** `coefficients` (list in the format of `decompose_image`); `wavelet` (`str`).

**Output.** `float64` image.

**How to write it yourself.** Return `np.asarray(pywt.waverec2(coefficients, wavelet, mode="periodization"), dtype=np.float64)`.

### Module `wavelets_lab.filters` (file `src/wavelets_lab/filters.py`)

Noise, linear smoothing (convolution with a mask) and median filtering of images, plus quality measures. Images follow the convention of `wavelets_lab.images` (`float64` on `[0, 255]`). Imports:
```python
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
```
The module defines two constants from the lecture notes: `GAUSS_3X3 = [[1, 2, 1], [2, 4, 2], [1, 2, 1]] / 16` and `GAUSS_5X5 = [[2, 4, 5, 4, 2], [4, 9, 12, 9, 4], [5, 12, 15, 12, 5], [4, 9, 12, 9, 4], [2, 4, 5, 4, 2]] / 159`. Both are integer approximations of a Gaussian, divided by the sum of their entries so that they sum to 1 (a constant image stays unchanged).

#### `filters.mse(reference, image)` (used internally)

**What it is.** Mean squared error between two images of the same shape: `mean((reference - image) ** 2)`.

**Input.** Two array-like images of the same shape.

**Output.** `float`.

**How to write it yourself.**
1. Convert both to `float64` and subtract.
2. Square, take `np.mean`, and convert to a Python `float`.

#### `filters.psnr(reference, image, peak=255.0)`

**What it is.** Peak signal-to-noise ratio in decibels, `10 * log10(peak**2 / MSE)`, the standard measure of image quality: higher is better; +6 dB roughly means half the error amplitude. Identical images give infinity.

**Input.**
- `reference` (array-like): the clean image.
- `image` (array-like): the image to evaluate, same shape.
- `peak` (`float`, default 255): largest possible intensity.

**Output.** `float` (dB), `inf` if the images are equal.

**How to write it yourself.**
1. `error = mse(reference, image)`.
2. If `error == 0`, return `float("inf")`.
3. Return `10 * np.log10(peak**2 / error)` as a `float`.

### Module `wavelets_lab.fourier` (file `src/wavelets_lab/fourier.py`)

Fourier series, the discrete Fourier transform (DFT) and Fourier coding of images. The DFT convention is the one of `numpy.fft`:
`X[k] = sum_{n=0}^{N-1} x[n] exp(-2 pi i k n / N)`, inverse `x[n] = (1/N) sum_k X[k] exp(2 pi i k n / N)`, so own implementations can be compared with the library directly. Only NumPy (and `collections.abc.Callable` for type hints) is imported.

#### `fourier.band_code(image, half_width)`

**What it is.** Band coding with a low-pass filter: all DFT coefficients outside a central square are set to zero and the image is reconstructed from the remaining ones. It is the simplest form of transform compression.

**Input.**
- `image` (array-like): greyscale image.
- `half_width` (`int`): half side of the kept square of frequencies.

**Output.** Tuple `(reconstruction, kept_fraction)`: the reconstructed `float64` image and the fraction of coefficients kept (e.g. `112**2 / 512**2 ≈ 0.048` for `half_width=56` and a 512×512 image).

**How to write it yourself.**
1. Centred spectrum: `spectrum = np.fft.fftshift(np.fft.fft2(image))`.
2. `mask = lowpass_mask(image.shape, half_width)`.
3. Zero the rest and invert: `np.fft.ifft2(np.fft.ifftshift(spectrum * mask)).real`.
4. Return the reconstruction and `float(mask.mean())` (the mean of a Boolean array is the fraction of `True`).

#### `fourier.lowpass_mask(shape, half_width)` (used internally)

**What it is.** A square low-pass mask for a centred spectrum: `True` on a `(2 * half_width) × (2 * half_width)` square around the centre (low frequencies), `False` elsewhere.

**Input.** `shape` (`tuple[int, int]`), `half_width` (`int`).

**Output.** Boolean array of shape `shape`.

**How to write it yourself.**
1. `mask = np.zeros(shape, dtype=bool)`; centre `(shape[0] // 2, shape[1] // 2)`.
2. Set `mask[c0 - half_width : c0 + half_width, c1 - half_width : c1 + half_width] = True`.